# 🏭 Real-Time Equipment Failure Detection — Model Training

This notebook generates synthetic industrial sensor data and trains an **Isolation Forest** anomaly detection model.

**Output**: `model.pkl` → download and place in your project's `model/` folder.

In [ ]:
!pip install scikit-learn pandas numpy -q
print('✅ Dependencies installed')

## Step 1: Generate Synthetic Sensor Data

We simulate 5,000 readings from three sensors:
- **Temperature (°C)**: Gaussian ~70 ± 2, with gradual drift
- **Vibration (G)**: Gaussian ~0.5 ± 0.05, with sudden jumps
- **Humidity (%)**: Sinusoidal ~40 ± 10, representing environmental cycles

2% of samples have **correlated anomaly spikes** injected.

In [ ]:
import pandas as pd
import numpy as np
import math
import pickle

np.random.seed(42)
N = 5000
WINDOW = 10
NOISE = 0.02

data = []
for i in range(N):
    drift = i * 0.001
    temp = np.random.normal(70, 2) + (drift * 5) + np.random.normal(0, NOISE * 5)
    vib = np.random.normal(0.5, 0.05) + np.random.normal(0, NOISE * 0.5)
    hum = 40.0 + 10.0 * math.sin(i / 100.0) + np.random.normal(0, NOISE * 2)
    
    if np.random.random() < 0.02:
        spike = np.random.uniform(20, 40)
        temp += spike
        vib += spike * 0.07
    
    data.append([temp, vib, hum])

df = pd.DataFrame(data, columns=['temperature', 'vibration', 'humidity'])
print(f'✅ Generated {len(df)} samples')
print(df.describe())

## Step 2: Sliding Window Feature Engineering

For each sensor, we compute 5 rolling features over a window of 10 readings:
- **Mean**: Average value
- **Std**: Volatility
- **Min / Max**: Extremes
- **Rate of Change**: Difference between newest and oldest

This transforms 3 raw features into a **15-dimensional vector**.

In [ ]:
SENSORS = ['temperature', 'vibration', 'humidity']

def extract_features(df, window_size=10):
    df = df.copy()
    for col in SENSORS:
        r = df[col].rolling(window=window_size, min_periods=1)
        df[f'{col}_mean'] = r.mean()
        df[f'{col}_std'] = r.std().fillna(0)
        df[f'{col}_min'] = r.min()
        df[f'{col}_max'] = r.max()
        df[f'{col}_roc'] = df[col].diff(periods=window_size-1).fillna(0)
    return df.dropna()

df_feat = extract_features(df, WINDOW)
feature_cols = [c for c in df_feat.columns if c not in SENSORS]
X = df_feat[feature_cols]

print(f'✅ Feature matrix shape: {X.shape}')
print(f'Features: {feature_cols}')

## Step 3: Train Isolation Forest

**Isolation Forest** is an unsupervised anomaly detection algorithm. It works by:
1. Randomly building decision trees
2. Anomalies require **fewer splits** to isolate → shorter path = more abnormal

Key parameters:
- `contamination=0.05` → Expect ~5% anomalies
- `n_estimators=100` → Number of trees

In [ ]:
from sklearn.ensemble import IsolationForest

clf = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
clf.fit(X)

scores = clf.score_samples(X)
predictions = clf.predict(X)

n_anomalies = (predictions == -1).sum()
print(f'✅ Model trained successfully!')
print(f'   Anomalies detected in training data: {n_anomalies} / {len(X)} ({n_anomalies/len(X)*100:.1f}%)')
print(f'   Score statistics: mean={scores.mean():.4f}, std={scores.std():.4f}')
print(f'   5th percentile threshold: {np.percentile(scores, 5):.4f}')

## Step 4: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Anomaly score distribution
axes[0,0].hist(scores, bins=50, color='steelblue', edgecolor='white')
axes[0,0].axvline(np.percentile(scores, 5), color='red', linestyle='--', label='5th percentile')
axes[0,0].set_title('Anomaly Score Distribution')
axes[0,0].legend()

# Temperature with anomalies highlighted
colors = ['red' if p == -1 else 'steelblue' for p in predictions]
axes[0,1].scatter(range(len(df_feat)), df_feat['temperature'], c=colors, s=3, alpha=0.5)
axes[0,1].set_title('Temperature (Red = Anomaly)')

# Vibration with anomalies highlighted
axes[1,0].scatter(range(len(df_feat)), df_feat['vibration'], c=colors, s=3, alpha=0.5)
axes[1,0].set_title('Vibration (Red = Anomaly)')

# Score over time
axes[1,1].plot(scores, color='steelblue', linewidth=0.5)
axes[1,1].axhline(np.percentile(scores, 5), color='red', linestyle='--')
axes[1,1].set_title('Anomaly Score Over Time')

plt.tight_layout()
plt.savefig('training_results.png', dpi=150)
plt.show()
print('✅ Plots saved to training_results.png')

## Step 5: Export Model

The exported `model.pkl` contains:
- Trained Isolation Forest model
- Feature column names
- Baseline score statistics (for adaptive thresholding)
- Reference data sample (for EvidentlyAI drift detection)

In [ ]:
output = {
    'model': clf,
    'features': feature_cols,
    'baseline_stats': {
        'mean_score': float(np.mean(scores)),
        'std_score': float(np.std(scores)),
        'threshold': float(np.percentile(scores, 5))
    },
    'reference_data': X.sample(500, random_state=42)
}

with open('model.pkl', 'wb') as f:
    pickle.dump(output, f)

print('✅ model.pkl exported!')
print('   Download this file and place it in your project\'s model/ directory.')